# 03_02 Trainable attention

*📖 Book: section 3.4*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
>
> **This notebook:** With nothing to learn, every letter mostly looks at itself. Query, key and value let the model learn what to look for, which is the attention step in `model.py`.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why simple attention isn't enough, and what query, key and value fix
- The library picture: what I look for, the label on each book, what's inside
- Why the scores are divided by the square root of 32
- How to rebuild the attention step of `model.py` and check it against the real model

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **query** | a list of numbers a letter makes to say what it is looking for | `Q = model.query(x)` |
| **key** | a list of numbers a letter makes to say what it offers, like the label on a book | `K = model.key(x)` |
| **value** | a list of numbers a letter makes to hand over when others pay attention to it, like what is inside the book | `V = model.value(x)` |

Words you've met before: **linear layer** ([01_02](../01_foundations/01_02_neural_networks.ipynb)).

## 3. The idea

### ❓ What was wrong with simple attention?

In [03_01](03_01_simple_attention.ipynb) every letter looked mostly at **itself**, because the same row was used for everything: for looking, for being looked at, and for the mix. With only one row, a letter can't learn "I want the vowel just before me".

So we give each letter three different rows, made from its embedding by three learned layers. Think of a library:

```
in a library                          in our model
what I am looking for (my request)  →  query  (Q)
the label on each book's spine      →  key    (K)
what is inside the book             →  value  (V)
```

Every letter is both a reader and a book. As a reader, it makes a query. As a book, it has a key (a label) and a value (its contents). A reader compares their query with every key (a dot product, which gives the score). Then they read the books that match well. And note that the mix is of the **values**, not of the labels.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: how are Q, K and V made?

By three **linear layers**. Each one is a table of learned weights that turns a row of 32 numbers into another row of 32. In `model.py` that's `self.query = nn.Linear(32, 32, bias=False)`, and the same for `key` and `value`. `bias=False` means the layer has no bias, the extra learned number a linear layer normally adds to each output ([01_02](../01_foundations/01_02_neural_networks.ipynb)). Attention works fine without it. The layers start out random and are learned in training, so the model finds out for itself what's worth asking, labelling and handing over.

We load our trained model (we only read it, we never save it) and look at the shapes for `ἄνδρα`.

👉 **Predict first:** the embedding rows `x` have shape `(5, 32)`. What shape will `Q`, `K` and `V` have? How many weights are in one layer, and in all three?

In [ ]:
import torch

from homer_ai.lessons import load_homer_model

# read only: we load the saved model, we never save
tokenizer, model = load_homer_model(fallback_untrained=True)

In [ ]:
ids = torch.tensor(tokenizer.encode("ἄνδρα"))
with torch.no_grad():
    x = model.embedding(ids)
    Q, K, V = model.query(x), model.key(x), model.value(x)

print("x", tuple(x.shape), " Q", tuple(Q.shape))
print("K", tuple(K.shape), " V", tuple(V.shape))
one_layer = model.query.weight.numel()
print("weights in one layer:", one_layer, " in all three:", 3 * one_layer)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

All four are `(5, 32)`: one row of 32 numbers per letter. Each layer has 32 × 32 = 1,024 weights, so that's 3,072 for the three. (Out of our model's 11,840 in all. The rest is the embedding, 4,384, and the `head`, 4,384.)

</details>

### ❓ Step 2: do queries and keys really change who looks at whom?

Yes. Here's a tiny example with the invented 2-number rows from 03_01 (`ἄ = [1, 0]`, `ν = [0, 1]`, `δ = [1, 1]`). Let the key be the row as it is, but let the **query** swap its two numbers (a learned layer could learn exactly this): `ἄ → [0, 1]`, `ν → [1, 0]`, `δ → [1, 1]`.

```
score of ν looking at:  ἄ: [1,0]·[1,0] = 1     ν: [1,0]·[0,1] = 0     δ: [1,0]·[1,1] = 1
```

👉 **Predict first:** in 03_01 each letter's top score was itself. Will `ν`'s top score still be itself?

In [ ]:
X = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
swap = torch.tensor([[0.0, 1.0], [1.0, 0.0]])  # a table that swaps the two numbers

tiny_Q = X @ swap  # queries: the rows with their two numbers swapped
tiny_K = X          # keys: the rows as they are
print(tiny_Q @ tiny_K.T)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Row `ν` is `[1, 0, 1]`. Its score with itself is now **0**, and it prefers `ἄ` and `δ`. So a query that differs from the key lets a letter look for something other than itself. And training finds a good table on its own.

</details>

**Look at:** the picture, starting from the left. `ν` is the reader. Its query (orange) is compared with each letter's key (green), the scores become shares, and the values (grey) are mixed by those shares. To keep it small, each value is just the letter's row as it is.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    GOOD,
    INK,
    MUTED,
    arrow,
    box,
    diagram_axes,
    studio_figure,
    tag,
)

# ν is the reader. The same tiny numbers as above: ν's query against the three keys.
scores = (tiny_Q @ tiny_K.T)[1]  # one score for each of ἄ, ν, δ
shares = torch.softmax(scores, dim=0)
mix = shares @ X  # the values are the rows as they are

fig = studio_figure("Query, key, value", "ν looks at ἄ, ν and δ")
ax = diagram_axes(fig)
for col, name, color in [
    (1.1, "query", ACCENT),
    (3.6, "key", GOOD),
    (5.0, "share", INK),
    (6.6, "value", MUTED),
    (9.0, "blend", INK),
]:
    tag(ax, col, 5.3, name, color, 10, weight="bold")

box(ax, 1.1, 2.9, 1.3, 0.8, "ν  Q", ACCENT, mono=True, lw=2.4)
tag(ax, 1.1, 2.15, "what ν asks for", MUTED, 8)
for i, letter in enumerate("ἄνδ"):
    y = 4.3 - 1.4 * i
    box(ax, 3.6, y, 1.2, 0.7, f"{letter}  K", GOOD, mono=True)
    arrow(
        ax, (1.75, 2.9), (3.0, y), f"{scores[i]:.0f}", ACCENT, 1.8,
        label_offset=(0, 0.22), fontsize=11,
    )
    arrow(ax, (4.25, y), (4.65, y), head=9)
    tag(ax, 5.0, y, f"{shares[i]:.2f}", INK, 11, mono=True)
    arrow(ax, (5.35, y), (5.95, y), head=9)
    box(ax, 6.6, y, 1.2, 0.7, f"{letter}  V", MUTED, mono=True)
    arrow(
        ax, (7.25, y), (8.3, 2.9), color=ACCENT if shares[i] > 0.4 else MUTED,
        width=1 + 9 * shares[i].item(), head=9,
    )
box(ax, 9.0, 2.9, 1.5, 0.9, f"[{mix[0]:.2f},\n {mix[1]:.2f}]", GOOD, mono=True, lw=2.4)
tag(ax, 3.1, 0.75, "score: query · key", MUTED, 9)
tag(ax, 6.1, 0.75, "share: softmax of the scores", MUTED, 9)
tag(ax, 8.9, 1.9, "mix of the values", MUTED, 8)
plt.show()

### ❓ Step 3: why divide the scores by the square root of 32?

The scores go into softmax, and softmax gets **too sharp** when the scores are big. One letter takes nearly all the share, the others get almost nothing, and training learns slowly (the model can't try out the other letters). A tiny example:

```
scores [1, 2, 3]     → shares?
scores [10, 20, 30]  → shares?     (same order, ten times bigger)
```

👉 **Predict first:** will the second list's shares be ten times different, or something more extreme?

In [ ]:
print(torch.softmax(torch.tensor([1.0, 2.0, 3.0]), dim=0))
print(torch.softmax(torch.tensor([10.0, 20.0, 30.0]), dim=0))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`[0.09, 0.24, 0.67]`, then `[0.0000, 0.0000, 1.0000]`. The biggest score takes everything.

</details>

### ❓ But why would our scores be big?

A dot product adds up one product per number, so more numbers (a longer row) means bigger scores. Random rows with 2 numbers give small scores, and rows with 512 numbers give huge ones. The score grows roughly like `√d`, where `d` is the row length (`√` is the square root: `√32` is about 5.7). Dividing by `√d` brings it back to a steady size.

Let's measure it with random numbers: 10,000 dot products for each row length `d`, and the typical size of a score (the standard deviation, a measure of spread).

👉 **Predict first:** for `d = 512`, about how big will a raw score be (is it near 1, or near `√512 ≈ 22.6`)?

In [ ]:
torch.manual_seed(0)
for d in [2, 32, 512]:
    q = torch.randn(10000, d)  # 10,000 random rows (random numbers around 0)
    k = torch.randn(10000, d)
    raw = (q * k).sum(dim=-1)  # 10,000 dot products
    scaled = raw / d**0.5
    print(f"d = {d:>3}: typical raw score {raw.std():5.1f}", end="   ")
    print(f"after dividing by √d {scaled.std():4.1f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The raw score grows with the row length (about 1.4, 5.7, 22.6, the same as `√d`), but after dividing by `√d` it stays near 1 every time. (`d**0.5` means "`d` to the power of one half", which is the square root.) Next, the effect on softmax itself: one query against 5 random keys, 1,000 times, and the average of the **biggest** share.

</details>

In [ ]:
torch.manual_seed(0)
for d in [2, 32, 512]:
    raw_top = scaled_top = 0.0
    for _ in range(1000):
        keys = torch.randn(5, d)
        query = torch.randn(d)
        s = keys @ query  # 5 scores
        raw_top += torch.softmax(s, dim=0).max().item() / 1000
        scaled_top += torch.softmax(s / d**0.5, dim=0).max().item() / 1000
    print(f"d = {d:>3}: biggest share, raw scores {raw_top:.2f}", end="   ")
    print(f"scaled scores {scaled_top:.2f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

With raw scores, the biggest share climbs towards 1 as `d` grows (0.51, 0.85, 0.96), so the softmax goes sharp. With scaled scores it stays about the same, near 0.45, whatever `d` is. That's why `model.py` divides by `x.shape[-1] ** 0.5`, which for us is `√32`.

</details>

### ❓ Step 4: can I rebuild the attention step of `model.py` myself?

Yes. We take the same `x`, `Q`, `K`, `V` as above and do each line ourselves, then peek inside the real model to compare. Scores first: every query against every key, divided by `√32`.

👉 **Predict first:** what shape will `scores` have, and what does the box in row 2, column 3 mean?

In [ ]:
scores = (Q @ K.T) / (x.shape[-1] ** 0.5)
print(tuple(scores.shape))
print(scores)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A `(5, 5)` table. The box in row 2, column 3 is how well the 2nd letter's query matches the 3rd letter's key. In other words, how much `ν` wants to listen to `δ`.

</details>

### ❓ Step 5: softmax, then the mix

Same as in 03_01, except that the scores now come from queries and keys, and the mix is of the **values**:

```
weights = softmax(scores)        (each row adds up to 1)
context = weights @ V            (a mix of the value rows)
```

In [ ]:
weights = torch.softmax(scores, dim=-1)
my_context = weights @ V
print("weights", tuple(weights.shape), " context", tuple(my_context.shape))
print("each row adds up to:", weights.sum(dim=-1))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`(5, 5)` weights, each row adding up to 1, and a `(5, 32)` context: one mixed row per letter.

</details>

### ❓ Does it match the real model?

`forward` only gives back the final scores, which are `model.head` applied to the context. So we check our context by passing it through `model.head` ourselves and comparing with what the real model returns. (`head` here is the last layer, not an attention head. Those come in 03_04.)

👉 **Predict first:** will `model.head(my_context)` equal the real model's scores?

In [ ]:
with torch.no_grad():
    real_scores = model(ids)  # the real model's scores, one row per letter
my_scores = model.head(my_context).detach()

print("same as the real model?", torch.allclose(my_scores, real_scores, atol=1e-5))
rows = [torch.allclose(my_scores[i], real_scores[i], atol=1e-5) for i in range(5)]
print("row by row:   ", rows)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

No, not the same: `False`. The row-by-row list shows that the first four rows differ. Only the last row matches, because the last letter is allowed to see all five letters anyway. We're missing one line from `model.py`: the **mask**, which hides the letters that come *after* each one. 03_03 explains it. Here we just copy the real line, so the check can finish.

</details>

In [ ]:
mask = torch.triu(torch.ones(5, 5) * float("-inf"), diagonal=1)  # explained in 03_03
weights = torch.softmax(scores + mask, dim=-1)
my_context = weights @ V
my_scores = model.head(my_context).detach()

print("same as the real model now?", torch.allclose(my_scores, real_scores, atol=1e-5))
rows = [torch.allclose(my_scores[i], real_scores[i], atol=1e-5) for i in range(5)]
print("row by row:", rows)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`True` for all five rows. Our step-by-step attention is the same as the one inside `HomerGreekLLM`, number for number.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why does the model need both a key and a value? Why not one row?

Because what makes a letter easy to **find** (its label) isn't always what we want to take from it (its contents). With two separate layers, the model can learn each on its own.

### ❓ Why is `bias=False` in the three layers?

A bias is one extra learned number added to every output. Attention works fine without it, so we leave it out. Fewer weights, same job.

## 5. Recap

- Three learned layers turn each embedding into a query (what I look for), a key (my label) and a value (my contents); scores are queries against keys.
- Scores are divided by √32 so softmax doesn't get too sharp: raw scores grow like √d with the row length, scaled ones stay near 1.
- Scores, softmax, then `weights @ V` rebuilds the real model's context exactly, once the mask is added.

**You can now:**

- say what the query, the key and the value are for
- compute trainable attention in three lines: scores, softmax, `weights @ V`

**Next:** [03_03: Hiding the future](03_03_hiding_the_future.ipynb): the mask line we just copied.